# Experiment 027 (S12): sweep-end generative evaluations, seeds 1 and 2 (Colab)

I fill the twelve missing generative evaluations for the p2 and p32 supervised arms
below 7B (007 at 0.5B and 3B, 010 at 1.5B). Evaluation only, no training. The
registered rules from the S12 plan apply: adapter inventory preflight before any
test-set contact, identity gate in scratch, transactional commit with the
evaluation manifest as the commit marker, null-field completion only for
`summary.json`, and a package halt on any integrity failure.

The first 40GB pass runs one 0.5B arm as a technical check. Continuation is
based only on successful execution and integrity checks, not on its accuracy.
After it commits, set `FIRST_ARM_ONLY = False` in the run cell. The GPU cell
then picks the phase: the 40GB card runs the
eight 007 arms, the 80GB card the four 010 arms, matching each cell's seed 0
per decision 6z. Finished evaluations skip, so a resumed session is safe.

## 0. Compute units and mount Drive

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before =   130.27       # fill from the resources panel
unit_rate = 6.77     # units per hour for this card
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '027_colab_s12_sweep_end_evals',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f)
        print('opened session record')

session 20260810_135746 | cu_before = 130.27
Mounted at /content/drive
opened session record


## 1. GPU check picks the session phase

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu, 'need an a100 runtime, both phases ran their seed 0 on a100'
if total_gb >= 79:
    PHASE = '80'
elif 35 <= total_gb <= 45:
    PHASE = '40'
else:
    raise AssertionError(f'unexpected vram {total_gb:.1f}, not a 40gb or 80gb a100')
print('phase:', PHASE, '| arms this session:',
      '007 at 0.5b and 3b' if PHASE == '40' else '010 at 1.5b')

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-ee879d14-049d-a814-1fe9-37a2dd06c53d)
total vram: 79.3 GiB
phase: 80 | arms this session: 010 at 1.5b


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3492, done.
remote: Counting objects: 100% (976/976), done.
remote: Compressing objects: 100% (684/684), done.
remote: Total 3492 (delta 598), reused 665 (delta 292), pack-reused 2516 (from 1)
Receiving objects: 100% (3492/3492), 108.57 MiB | 18.09 MiB/s, done.
Resolving deltas: 100% (2217/2217), done.
Updating files: 100% (1585/1585), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml

# colab preinstalls a torchao that conflicts with this peft version
# no quantization is used here
!pip uninstall -q -y torchao
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 151.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 48.8 MB/s eta 0:00:00
deps installed


## 4. The arm table and the environment assert

In [5]:
# the twelve arms live in scripts/s12_eval.py next to the machinery the tests
# cover. Tags derive from the configs so directory names cannot drift
import json, os
from config import load_config, run_tag
from scripts.s12_eval import ARMS

D = '/content/drive/MyDrive/inm363'

def arm_items(cfg_path, seed):
    cfg = load_config(cfg_path)
    cfg.train.seed = seed
    cfg.eval.skip_generative = False
    tag = run_tag(cfg)
    return cfg, tag, os.path.join(D, 'results', tag), os.path.join(D, 'checkpoints', tag, 'sft')

ALL_ARMS = ARMS['40'] + ARMS['80']

# library versions and dtype must match what every parent run recorded, not
# just the first one
from obs import _versions
cur = _versions()
pinned = ['trl', 'transformers', 'peft', 'accelerate', 'torch']
for cfg_path, seed in ALL_ARMS:
    cfg, tag, rdir, adir = arm_items(cfg_path, seed)
    parent = json.load(open(os.path.join(rdir, 'manifest.json')))
    vers = parent.get('versions', {})
    for lib in pinned:
        if lib in vers and lib in cur:
            assert cur[lib] == vers[lib], f'{tag} {lib}: {cur[lib]} vs parent {vers[lib]}'
    dt = parent.get('resolved_dtype')
    if dt is not None:
        assert dt == 'bfloat16', f'{tag}: parent dtype {dt}'
print('all twelve parent manifests agree with this environment')

all twelve parent manifests agree with this environment


## 5. Preflight: adapter inventory for all twelve arms, before any evaluation

In [6]:
# every adapter must exist, carry the training-time file set, and hash-match its
# config before anything touches the test set. Any failure halts the package here
from scripts.s12_eval import ADAPTER_FILES, config_hash_status, sha256

PREFLIGHT = {}
CONFIG_HASH_STATUS = {}
problems = []
for cfg_path, seed in ALL_ARMS:
    cfg, tag, rdir, adir = arm_items(cfg_path, seed)
    missing = [n for n in ADAPTER_FILES if not os.path.exists(os.path.join(adir, n))]
    if missing:
        problems.append(f'{tag}: missing {missing}')
        continue
    stored = open(os.path.join(adir, 'config_hash.txt')).read().strip()
    parent = json.load(open(os.path.join(rdir, 'manifest.json')))
    hash_status = config_hash_status(cfg, parent, stored)
    if hash_status == 'mismatch':
        problems.append(f'{tag}: config hash mismatch')
        continue
    probe_p = os.path.join(rdir, 'sft_logprob_preds.jsonl')
    if not os.path.exists(probe_p) or sum(1 for l in open(probe_p) if l.strip()) != 1884:
        problems.append(f'{tag}: archived probe predictions missing or short')
        continue
    PREFLIGHT[tag] = {n: sha256(os.path.join(adir, n)) for n in ADAPTER_FILES}
    CONFIG_HASH_STATUS[tag] = hash_status
    print('ok:', tag, '| config hash:', hash_status)

assert not problems, 'preflight halt, resolve via a separately registered contingency:\n' + '\n'.join(problems)
print('all twelve adapters present, hash-bound and gate-ready')

ok: 007_p2_COLAB_FULL_B05_sft_only_seed1 | config hash: legacy_missing_sft_from_adapter
ok: 007_p2_COLAB_FULL_B05_sft_only_seed2 | config hash: legacy_missing_sft_from_adapter
ok: 007_p32_COLAB_FULL_B05_sft_only_seed1 | config hash: legacy_missing_sft_from_adapter
ok: 007_p32_COLAB_FULL_B05_sft_only_seed2 | config hash: legacy_missing_sft_from_adapter
ok: 007_p2_COLAB_FULL_B3_sft_only_seed1 | config hash: legacy_missing_sft_from_adapter
ok: 007_p2_COLAB_FULL_B3_sft_only_seed2 | config hash: legacy_missing_sft_from_adapter
ok: 007_p32_COLAB_FULL_B3_sft_only_seed1 | config hash: legacy_missing_sft_from_adapter
ok: 007_p32_COLAB_FULL_B3_sft_only_seed2 | config hash: legacy_missing_sft_from_adapter
ok: 010_p2_COLAB_FULL_B15_sft_only_seed1 | config hash: current
ok: 010_p2_COLAB_FULL_B15_sft_only_seed2 | config hash: current
ok: 010_p32_COLAB_FULL_B15_sft_only_seed1 | config hash: current
ok: 010_p32_COLAB_FULL_B15_sft_only_seed2 | config hash: current
all twelve adapters present, hash-boun

## 6. Build the data splits (only after the preflight passed: this touches the test split)

In [7]:
!python scripts/build_data.py

README.md: 100% 7.93k/7.93k [00:00<00:00, 22.5MB/s]

main/train-00000-of-00001.parquet: downloading bytes:  61% 1.41M/2.31M [00:01<00:00, 1.07MB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.51MB/s,  220kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.51MB/s,  223kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 309kB/s, 40.7kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 309kB/s, 40.7kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 363885.83 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 285493.19 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 65.2MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 133MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00

## 7. Wiring check

In [8]:
# the transactional machinery has its own test file, run it before anything else
!python -m pytest tests/test_s12_sweep_end_evals.py -q

from eval.harness import run_eval, run_logprob_eval
from scripts.env_switch import resolve_device
from scripts.s12_eval import GateFailure, IntegrityError, run_arm
from obs import append_ledger, gpu_snapshot, _git_info
from train.model_utils import resolve_dtype
print('imports fine | phase', PHASE, '| arms:',
      [arm_items(c, s)[1] for c, s in ARMS[PHASE]])

................                                                         [100%]
16 passed in 0.44s
imports fine | phase 80 | arms: ['010_p2_COLAB_FULL_B15_sft_only_seed1', '010_p2_COLAB_FULL_B15_sft_only_seed2', '010_p32_COLAB_FULL_B15_sft_only_seed1', '010_p32_COLAB_FULL_B15_sft_only_seed2']


## 8. The run: identity gate, generative evaluation, transactional commit

In [9]:
# both cost fields must be filled before anything runs
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
import os

SCRATCH = '/content/s12_scratch'
LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')

# run one arm first to check the full path, then set this to false
FIRST_ARM_ONLY = False
arms_to_run = ARMS[PHASE][:1] if FIRST_ARM_ONLY else ARMS[PHASE]
print('selected arms:', [arm_items(c, s)[1] for c, s in arms_to_run])

for cfg_path, seed in arms_to_run:
    cfg, tag, rdir, adir = arm_items(cfg_path, seed)
    print('=== ' + tag + ' ===')
    device = resolve_device(cfg.env)
    sdir = os.path.join(SCRATCH, tag)

    def gen_eval(cfg=cfg, device=device, adir=adir, sdir=sdir):
        return run_eval(cfg, device, adapter_path=adir, condition='sft', out_dir=sdir)

    def gate_eval(cfg=cfg, device=device, adir=adir, sdir=sdir):
        run_logprob_eval(cfg, device, adapter_path=adir, condition='gate', out_dir=sdir)

    def bill(timings, done, tag=tag):
        append_ledger(LEDGER, {'run_tag': tag, 'condition': 's12_eval',
                               'timings_sec': timings, 'train_sec': 0, 'train_steps': 0,
                               'completed': done, 'gpu': gpu_snapshot()})

    extra = {'adapter': adir, 'adapter_sha256': PREFLIGHT[tag], 'config': cfg_path,
             'config_hash_status': CONFIG_HASH_STATUS[tag],
             'seed_override': seed, 'git': _git_info(), 'versions': _versions(),
             'resolved_dtype': resolve_dtype(cfg), 'gpu': gpu_snapshot()}
    try:
        status = run_arm(rdir, sdir, gen_eval, gate_eval, extra_manifest=extra, bill=bill)
    except (GateFailure, IntegrityError) as e:
        raise SystemExit(f'{e}\npackage halts and stays nonbinding per the plan')
    print(' ', status)

print('selected arms done:', len(arms_to_run), 'of', len(ARMS[PHASE]))
if FIRST_ARM_ONLY:
    print('first-arm check complete. Set FIRST_ARM_ONLY = False and rerun this cell.')

selected arms: ['010_p2_COLAB_FULL_B15_sft_only_seed1', '010_p2_COLAB_FULL_B15_sft_only_seed2', '010_p32_COLAB_FULL_B15_sft_only_seed1', '010_p32_COLAB_FULL_B15_sft_only_seed2']
=== 010_p2_COLAB_FULL_B15_sft_only_seed1 ===


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (23 outputs left out)

  gate: {'agreement': 1.0, 'acc_new': 0.698514, 'acc_archived': 0.698514}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

sft eval (1884 items):   0%|          | 0/118 [00:00<?, ?it/s]

  committed
selected arms done: 4 of 4


## 9. Peek at the results

In [10]:
# quick look: the new seeds next to the archived seed 0, per arm family
for cfg_path, seed in ARMS[PHASE]:
    cfg, tag, rdir, adir = arm_items(cfg_path, seed)
    if os.path.exists(os.path.join(rdir, 'sft_eval.json')):
        ev = json.load(open(os.path.join(rdir, 'sft_eval.json')))
        cfg0, tag0, rdir0, _ = arm_items(cfg_path, 0)
        ev0 = json.load(open(os.path.join(rdir0, 'sft_eval.json')))
        print(tag)
        print('  seed', seed, ev['per_tier_acc'])
        print('  seed 0 ', ev0['per_tier_acc'])
    else:
        print(tag, ': not evaluated yet')

010_p2_COLAB_FULL_B15_sft_only_seed1
  seed 1 {'easy': 0.7781569965870307, 'medium': 0.6568144499178982, 'hard': 0.44696969696969696}
  seed 0  {'easy': 0.7940841865756542, 'medium': 0.6600985221674877, 'hard': 0.4090909090909091}
010_p2_COLAB_FULL_B15_sft_only_seed2
  seed 2 {'easy': 0.78839590443686, 'medium': 0.6600985221674877, 'hard': 0.4116161616161616}
  seed 0  {'easy': 0.7940841865756542, 'medium': 0.6600985221674877, 'hard': 0.4090909090909091}
010_p32_COLAB_FULL_B15_sft_only_seed1
  seed 1 {'easy': 0.7986348122866894, 'medium': 0.6896551724137931, 'hard': 0.4494949494949495}
  seed 0  {'easy': 0.8077360637087599, 'medium': 0.6929392446633826, 'hard': 0.44696969696969696}
010_p32_COLAB_FULL_B15_sft_only_seed2
  seed 2 {'easy': 0.7952218430034129, 'medium': 0.7060755336617406, 'hard': 0.44696969696969696}
  seed 0  {'easy': 0.8077360637087599, 'medium': 0.6929392446633826, 'hard': 0.44696969696969696}


## 10. Close the units record

In [12]:
# closing balance, completes the open record for this session
import json, os, time
cu_after =  126.3      # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

no open record for this session, it may already be closed
